# Lab 3: Hashes to ashes

As usual, start by writing <b style="color:red">HACKOOLIQUES</b> here.

## A) Pearson hashing

You are given a permutation of the set of 2-byte strings as a Python dictionary:

In [25]:
import pickle

with open('perm', 'rb') as file:
    perm = pickle.load(file)

If you want to know the image of, say, `3c2d` under this permutation, just look at the corresponding entry: 

In [9]:
perm[bytes.fromhex("0002")].hex()
perm[bytes.fromhex("3c2d")].hex()

'bf21'

Use this permutation to construct a Pearson hash function turning arbitrarily long byte arrays into 2-byte hashes (you may assume the input contains an even number of bytes to avoid padding issues).

Make sure that you get the following values:

$m = \verb|b"hello!"|$, $h = \tt{1b3f}$

$m = \verb|b"HELLO?"|$, $h = \tt{fea5}$

$m = \verb|b"A longer one"|$, $h = \tt{e1ba}$

In [32]:
def pearson(B, perm):
    out = b'\x00\x00'
    for i in range(0, len(B), 2):
        pair = B[i:i+2]
        out = perm[bytes(a ^ b for a, b in zip(out, pair))]
    return out


In [34]:
M = [b"hello!",b"HELLO?",b"A longer one"]
for m in M:
    print(pearson(m, perm).hex())

1b3f
fea5
e1ba


## B) Birthday attack

From now on, you play the attacker and try to break the above (insecure) hash function, to which you have unlimited access (*i.e.*, you can compute as many hashes as you want with it).

How many hashes of randomly generated strings do you _expect_ it would take before a collision is found for this hash function?

2 octets = 16 bits -> 2^16 = N = taille de l’espace de clés = 65 536

- p≈1−exp(−k(k−1)/2N​)
- k≈sqrt(2Nln(1/(1−p)​))

| Probabilité (p) | Nombre d’essais (k) (≈) |
| --------------: |  ----------------------: |
|             25% |               **195** |
|             50% |               **302** |
|             75% |              **427** |
|             99% |              **777** |


Check how lucky you are today by performing a birthday attack on the Pearson hash function. How many hashes did you actually need to compute? (compare with the expected value and discuss)

In [ ]:
import math
import os
import random
import time
import statistics


def random_message(min_len=6, max_len=32) -> bytes:
    L = random.randint(min_len, max_len)
    if L%2 == 1 : return random_message()
    return os.urandom(L)

def find_collision_once(perm, max_attempts=2_000_000, report_every=0):
    seen = {}
    start = time.perf_counter()
    for attempt in range(1, max_attempts + 1):
        msg = random_message()
        h = pearson(msg, perm)
        prev = seen.get(h)
        if prev is None:
            seen[h] = msg
        elif prev != msg:
            elapsed = time.perf_counter() - start
            return attempt, elapsed
        if report_every and (attempt % report_every == 0):
            print(f"[{attempt:,}] essais — table {len(seen):,} — {time.perf_counter()-start:.1f}s")
    return None  # pas de collision trouvée

def benchmark(perm: dict, n_trials: int = 100, max_attempts: int = 200_000):
    results = []
    total_start = time.perf_counter()

    for _ in range(n_trials):
        r = find_collision_once(perm, max_attempts=max_attempts)
        if r is None:
            continue
        attempts, _ = r
        results.append(attempts)

    total_elapsed = time.perf_counter() - total_start

    if not results:
        return {
            "n_trials": n_trials,
            "completed": 0,
            "total_time_s": total_elapsed
        }

    results.sort()
    def percentile(lst, pct):
        n = len(lst)
        pos = (pct / 100) * (n - 1)
        lo = int(pos)
        hi = int(math.ceil(pos))
        if lo == hi:
            return lst[lo]
        return lst[lo] * (1 - (pos - lo)) + lst[hi] * (pos - lo)

    percentiles = {p: percentile(results, p) for p in [25, 50, 75, 99]}

    return {
        "n_trials": n_trials,
        "completed": len(results),
        "mean": statistics.mean(results),
        "median": statistics.median(results),
        "stdev": statistics.stdev(results) if len(results) > 1 else 0.0,
        "percentiles": percentiles,
        "total_time_s": total_elapsed
    }



stats = benchmark(perm, n_trials=10000, max_attempts=200000)
print(stats)


{'n_trials': 10000, 'completed': 10000, 'mean': 321.7323, 'median': 303.0, 'stdev': 166.8133415102059, 'percentiles': {25: 195.0, 50: 303.0, 75: 428.0, 99: 777.0100000000002}, 'total_time_s': 21.80338169999959}


Sur 10k tentatives : moyenne ~320 tentatives, mediane 302
{'n_trials': 10000, 'completed': 10000, 'mean': 321.7323, 'median': 303.0, 'stdev': 166.8133415102059, 'percentiles': {25: 195.0, 50: 303.0, 75: 428.0, 99: 777.0100000000002}, 'total_time_s': 21.80338169999959}

Sur 10000 tentatives (grand nombre):
| Probabilité (p) | Nombre d’essais (k) (≈) | Obtenu |
| --------------: |  ----------------------: | ----: |
|             25% |              195 | 195.0 |
|             50% |              302 | 303.3 |
|             75% |              427 | 427.0 |
|             99% |              777 | 777.01 |


Les résultats obtenus sont quasiment identiques à ceux théoriques pour une birthday attack sur 16 bits. Ces résultats montrent que les collisions surviennent beaucoup plus tôt que la taille de l'espace total ( 777 tentatives sur un espace de 65563 ).

## C) Extension attack

The situation is much worse than that! since the Pearson hash function is not designed to be cryptographically secure. Convince yourself that, given two strings, such as $a = \verb|"target"|$ and $b = \verb|"beginning of the end"|$, it is always possible to append two characters at the end of $b$ to get a new string $b'$ such that $H(b') = H(a)$. Be fair game by using only information known to the attacker (<i>i.e.</i>, it is forbidden to look at $\tt{perm}$ but you can perform as many hash evaluations as you like).

In [56]:
def extension_brute_force(perm, a, b):
    p_a = pearson(a, perm)
    for i in range(256):
        for j in range(256):
            trial = b + bytes([i, j])
            if pearson(trial, perm) == p_a:
                # print("Brute force:", bytes([i, j]), f"{j + (i*256)} tentatives")
                return  bytes([i, j]), j + (i*256)
            
def extension_birthday(perm, a, b, max_attempts=200000):
    p_a = pearson(a, perm)
    for attempt in range(1, max_attempts + 1):
        s = os.urandom(2)               # suffixe aléatoire de 2 octets
        if pearson(b + s, perm) == p_a:
            # print("Birthday:", s, f"{attempt} tentatives")
            return s, attempt


In [62]:
a=b"target"
b=b"beginning of the end"

s,t = extension_brute_force(perm,a,b)
print(f"Brute force: {s.hex()} ({t} tentatives)", )


Brute force: bc81 (48257 tentatives)


In [ ]:

import os, time, statistics, math, random

def percentile(lst, pct):
    s = sorted(lst)
    if not s:
        return None
    n = len(s)
    pos = (pct/100.0) * (n-1)
    lo = int(math.floor(pos)); hi = int(math.ceil(pos))
    if lo == hi:
        return s[lo]
    frac = pos - lo
    return s[lo] * (1-frac) + s[hi] * frac

def simple_benchmark(a, b, perm, n_trials=50, max_attempts=200_000):
    bf_attempts = None
    bd_attempts = []
    bf_times = None
    bd_times = []
    
    # brute force (temps constant)
    t0 = time.perf_counter()
    res = extension_brute_force(perm, a, b)
    dt = time.perf_counter() - t0
    if res:
        _, attempts = res
        bf_attempts = attempts
        bf_times = dt
        
    for t in range(n_trials):
        # birthday sampling
        t0 = time.perf_counter()
        res = extension_birthday(perm, a, b, max_attempts=max_attempts)
        dt = time.perf_counter() - t0
        if res:
            _, attempts = res
            bd_attempts.append(attempts)
            bd_times.append(dt)

    def stats(lst):
        if not lst:
            return None
        return {
            "n": len(lst),
            "mean": statistics.mean(lst),
            "median": statistics.median(lst),
            "stdev": statistics.stdev(lst) if len(lst) > 1 else 0.0,
            "p25": percentile(lst, 25),
            "p50": percentile(lst, 50),
            "p75": percentile(lst, 75),
            "p99": percentile(lst, 99)
        }

    return {
        "bruteforce_attempts": bf_attempts,
        "bruteforce_times_s": bf_times,
        "birthday_attempts": stats(bd_attempts),
        "birthday_times_s": stats(bd_times),
    }


a = b"target"
b = b"beginning of the end"
OUT = simple_benchmark(a, b, perm, n_trials=100, max_attempts=200_000)
print("Brute-force attempts:", OUT["bruteforce_attempts"])
print("Birthday attempts   :", OUT["birthday_attempts"])
print("Brute-force times s :", OUT["bruteforce_times_s"])
print("Birthday times s    :", OUT["birthday_times_s"])


Brute-force attempts: 48257
Birthday attempts   : {'n': 94, 'mean': 53182.84042553192, 'median': 37245.0, 'stdev': 45162.57330099571, 'p25': 14978.75, 'p50': 37245.0, 'p75': 94264.25, 'p99': 164591.68999999994}
Brute-force times s : 0.26836509999884584
Birthday times s    : {'n': 94, 'mean': 0.29230259255320484, 'median': 0.20186269999976503, 'stdev': 0.2493121272659202, 'p25': 0.0816133749999608, 'p50': 0.20186269999976503, 'p75': 0.5056839999992917, 'p99': 0.9385694610009522}


Complexite 2^8 * 2^8 = 2^16

Il suffit de tester au plus 65536 évaluations de hachage pour trouver deux octets xy tels que H(b||xy)=H(a). Dans l’expérience, un suffixe valide a été trouvé en ≤65 536 essais — faisable par bruteforce.

**Comparaison avec Birthday attack**:

Brute-force attempts (constant): 48257, 0.268s

| Metric | Birthday Attempts | Birthday Times (s) |
| ------ | ----------------- | ------------------ |
| n      | 94                | 94                 |
| mean   | 53182.84          | 0.2923             |
| median | 37245.0           | 0.2019             |
| stdev  | 45162.57          | 0.2493             |
| p25    | 14978.75          | 0.0816             |
| p50    | 37245.0           | 0.2019             |
| p75    | 94264.25          | 0.5057             |
| p99    | 164591.69         | 0.9386             |

Les deux méthodes restent faisables ici. L’exhaustif offre une constance temporelle tandis que le sampling peut être parfois sensiblement plus lent.

What's the bit complexity of your attack? Try to make it as efficient as possible.

[ _Hint_: it's easy to make it in $2^{16}$ steps, but with a bit of cleverness you can take it down to $2^0$... ]

In [63]:
inv_perm = {v: k for k, v in perm.items()}

p_a = pearson(a, perm)
p_b = pearson(b, perm)
k = inv_perm[p_a]
s = bytes(x ^ y for x,y in zip(p_b, k))

p_a2 = pearson(b+s, perm)

print(s.hex(), p_a == p_a2 )


bc81 True


L’inverse de perm est construit avec un coût fixe afin de retrouver rapidement, pour tout hash cible p_a, la clé k telle que perm[k]=p_a.
Le suffixe s s’obtient en XORant k avec l’état intermédiaire p_b de b, ce qui garantit pearson(b||s)=p_a en une évaluation.

## D) Dictionary/brute-force attack

Now a "real-world" (broken) example. Some hackers have leaked a large database of hashed password from a well-known online retailer, where we can read hashed passwords:

```
Alice  753692ec36adb4c794c973945eb2a99c1649703ea6f76bf259abb4fb838e013e
Bob    751bc918ef403f5b898e321b6266b3e7f8fc8decfcc9a83bb83ce771527a828d
Carol  f7092ca05af91105bfdfb64b29196420f936b2f01dd91d62355c549e44312f26
David  e67ad6e81a23071f87a7edb3b45752d4b54e3d41d2b23d05038db23b1c2802fd  
...
```

Noticing that they are 64 hexadecimal digits long and that the value next to Alice's name is the SHA-256 of `Hallo` (you should be able to check that!), you suspect that these are just unsalted SHA-256 hashes. 

You think Bob might be the kind of person to use his birthdate as password, and you know he was born after 2000. Can you recover his password? What's the bit complexity of your attack?

In [28]:
import hashlib

Alice="753692ec36adb4c794c973945eb2a99c1649703ea6f76bf259abb4fb838e013e"
Bob="751bc918ef403f5b898e321b6266b3e7f8fc8decfcc9a83bb83ce771527a828d"
Carol="f7092ca05af91105bfdfb64b29196420f936b2f01dd91d62355c549e44312f26"
David="e67ad6e81a23071f87a7edb3b45752d4b54e3d41d2b23d05038db23b1c2802fd"

def sha256(m):
    return hashlib.sha256(m.encode()).digest().hex()

print("Hallo == sha256(Alice):", sha256("Hallo") == Alice)

for y in range(2000,2026):
    for m in range(1,13):
        for d in range(1,32):
            date_str = f'{m:02}/{d:02}/{y:04}'
            if sha256(date_str) == Bob : 
                print("Bob:", date_str)
                break

Hallo == sha256(Alice): True
Bob: 03/25/2008


Le sha256(Hallo) correspond bien au hash associé à Alice, en l'absence de sel, et en présence de certaines informations, il est simple de remonter jusqu'au mot de passe d'origine.

En générant toutes les dates possibles du 01/01/2000 à aujourd'hui ( ~31 x 12 x 26 = 9672 dates ~2^13 bits) et en calculant le sha256, il est possible de remonter jusqu'au mot de passe de Bob.


[ Bonus points if you recover Carol's secret password as well! ]

In [30]:
with open("../rockyou.txt.sha256.txt") as f:
    for line in f.readlines():
        hash,pwd = line.split(":",1)
        if hash == Carol:
            print("Carol:", pwd)
            break

Carol: 708xy95BobaFett



A partir d'une rainbowtable ( fichier hash:mdp précalculé ), il est possible de retrouver le mot de passe de Carol.
Avec un gros investissement initial en calcul de hash O(n) et en stockage O(n), et si le dictionnaire est cahrgé en mémoire, il est possible de retrouver le mot de passe initial en O(1) avec l'hypothèse d'aucune collision.   

Attaque par rainbow table, ici le dictionnaire n'est pas en mémoire, c'est une lecture en O(n)
(18.5) s Carol: 708xy95BobaFett

[ Extra bonus points if you recover David's secret password as well! Good luck though... ]

En l'absence de social engineering ou d'entrée dans la rainbow table, "impossible" de retrouver le pwd.
Il est possible d'essayer plusieurs motifs classiques de mot de passe, d'itérer sur les dates de naissance avec les hypothèses sur son âge, mais sans information sur la nature du mot de passe c'est quasiment impossible.